# ML-03 - Frame My Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhalid04/Shaheer-Khalid-FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Week 1 I picked Lane 2, Refresh / Content Opportunity Scoring, and wrote the question in plain words. This notebook does the next bit: turning that question into a shape a model can actually be fitted to. Task type, target, metric, unit of analysis, and whether a model is even the right tool.

The one-paragraph frame, up front:

> For a **content editor** deciding **which pages to open first this week**, I will build a **ranked queue with reason codes** from **per-page search and engagement history**, scoring **the probability that a page loses more than 20% of its search demand in the next 30 days**, measured by **precision@K against the best single-signal rule**. A wrong call costs **an editor's hour spent on a page that was fine, or a page with real demand quietly bleeding while nobody looks**. A plain rule isn't enough because **the obvious one-line rules pick almost completely different pages, and two of the five are worse than doing nothing**. I will claim only **observed, directional, decision-support** results.

Everything below is me checking each piece of that paragraph against the actual data.

In [1]:
# Setup. The loop walks up to the repo root so this runs in Colab or locally.
import os

import numpy as np
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns, {df['client_id'].nunique()} clients")

30,000 rows x 44 columns, 32 clients


## 1. My lane as an ML task (type)

**Ranking, delivered as a score.** Under the hood I will fit a supervised binary classifier, but the thing I care about is the predicted probability used as a sort key, then cut at K = however many pages an editor can actually review. So the model gets trained like a classifier and judged like a ranker.

Why ranking and not the other three:

- **Not plain classification.** A yes/no verdict is nearly useless here, and the cell below shows why: among pages with real demand, about 62% already fit "losing ground". A flag that lights up on 12,000 pages out of 20,000 does not help anyone decide what to open at 9am on Monday. The editor's constraint is not "which pages are declining", it is "I have six hours". Only an order answers that.
- **Not clustering.** Clustering would tell me what kinds of pages exist, which is genuinely interesting, but it has no capacity constraint in it and no way to say this one before that one. It is a lens, not a queue.
- **Not pure signal analysis.** That stops at "these things move together" and hands the editor no list.

The giveaway is the shape of the question itself. "Which ones first?" is the ranking row of the task-type table, and my Week 1 question is literally that sentence.

One thing I want to be honest about: calling it ranking does not mean I will use a learning-to-rank algorithm. I will use ordinary models and treat their probability output as the score. The word "ranking" is about how I *evaluate* it, which is precision@K, not about which library I import.

In [2]:
# Why a yes/no flag doesn't answer the editor's question.
# Eligible = pages with enough demand in the earlier window that a drop would be worth someone's time.
eligible = df[df["impressions_prev_30d"] >= 50].copy()

# Sketch of the target, defined properly in section 2. Here just to size the problem.
sketch = (eligible["impressions_last_30d"] < 0.8 * eligible["impressions_prev_30d"]).astype(int)

n_flagged = int(sketch.sum())
capacity_per_week = 50

print(f"Pages with demand:        {len(eligible):,}")
print(f"Would be flagged 'yes':   {n_flagged:,}  ({sketch.mean() * 100:.1f}% of them)")
print(f"At {capacity_per_week} reviews/week that flag is a {n_flagged / capacity_per_week:,.0f}-week backlog.")
print("so the useful output is the ORDER of that list, not the flag. ranking, not classification.")

Pages with demand:        20,249
Would be flagged 'yes':   12,455  (61.5% of them)
At 50 reviews/week that flag is a 249-week backlog.
so the useful output is the ORDER of that list, not the flag. ranking, not classification.


## 2. Target or proxy

**What I want to predict:** for a page that has real search demand today, does it lose more than 20% of that demand over the next 30 days?

Written as a rule I could hand to someone else:

```text
unit:     one page, observed at a cutoff date T
eligible: impressions in [T-29, T] >= 50
label:    y = 1 if impressions in [T+1, T+30] < 0.8 x impressions in [T-29, T]
features: only columns computed from data on or before T
```

**And this is where I have to be straight about the starter file: what is in it is a proxy, not an outcome, and it is a leaky one.**

Two separate problems, and I kept mixing them up in my head, so I am writing them apart:

1. **It is defined, not observed.** `is_declining_label` is `trend_direction == "down"`, and `trend_direction` is just buckets cut out of `trend_pct`. Nobody measured it happening, someone wrote a threshold. A model trained on it learns the threshold, not the world. The cell below shows my sketched label agreeing with `trend_direction == "down"` on 100% of eligible rows, which is not a nice result, it is the proof that the starter label is a restatement of a rule I already know.
2. **The windows overlap, so the features contain the answer.** This is the one I nearly missed. Every 90-day column in this file, `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, spans a window that *includes* the last 30 days. Those last 30 days are exactly the outcome period the label is cut from. So `impressions_90d` carries a median 22% of the outcome window inside it, and correlates 0.92 with it. Even without touching `trend_pct` or `trend_direction`, the ordinary-looking features leak.

Which means the starter slice cannot give me an honest target, and no amount of dropping the two banned columns fixes it. The problem is the grain, not the column list.

**So the capstone moves to `fact_content_daily_performance` in the warehouse release.** Daily rows let me cut the windows myself: features from `[T-90, T]`, label from `[T+1, T+30]`, no overlap by construction. Per the data skill I will set T per client rather than using one global date, because history depth differs a lot between clients, and I will develop the label logic on a mid-panel month, not on the `_sample` table, since that sample is the final month and is exactly where my test window needs to live.

For this notebook the sketch below is the right *shape* (an earlier window predicting a later one) drawn on the only two windows the starter file gives me. I am using it to size the problem and to demonstrate the leak, not as something I would train on.

In [3]:
# The target, sketched. Same SHAPE as the capstone label: earlier window -> later window.
work = df[df["impressions_prev_30d"] >= 50].copy()
work["y_demand_loss_30d"] = (
    work["impressions_last_30d"] < 0.8 * work["impressions_prev_30d"]
).astype(int)

base_rate = work["y_demand_loss_30d"].mean()
print(f"Eligible pages (>=50 impressions in the earlier 30d): {len(work):,}")
print(f"Base rate of demand loss:                            {base_rate * 100:.1f}%")

# Problem 1: the starter label is this rule restated, not an observed outcome.
agreement = (work["y_demand_loss_30d"] == (work["trend_direction"] == "down").astype(int)).mean()
print(f"Agreement with trend_direction == 'down':            {agreement * 100:.1f}%")

# Problem 2: the 90-day feature columns CONTAIN the outcome window.
share_inside = (work["impressions_last_30d"] / work["impressions_90d"].replace(0, np.nan)).median()
corr_leak = work["impressions_90d"].corr(work["impressions_last_30d"])
print()
print(f"Median share of impressions_90d that IS the outcome window: {share_inside * 100:.1f}%")
print(f"Correlation impressions_90d vs impressions_last_30d:        {corr_leak:.2f}")
print("so even the innocent-looking 90d columns leak. the fix is daily rows, not a shorter feature list.")

Eligible pages (>=50 impressions in the earlier 30d): 20,249
Base rate of demand loss:                            61.5%
Agreement with trend_direction == 'down':            100.0%

Median share of impressions_90d that IS the outcome window: 22.2%
Correlation impressions_90d vs impressions_last_30d:        0.92
so even the innocent-looking 90d columns leak. the fix is daily rows, not a shorter feature list.


## 3. Success metric

**precision@K, with K set to what an editor can actually review, judged against two floors.**

K is not a hyperparameter, it is a staffing fact. If a team reviews 50 pages a week, then precision@1000 is a number about nothing. I will report at K = 50, 100 and 200 and lead with whichever matches the capacity I am given.

The two floors matter more than the headline number, because with a 62% base rate a queue can look excellent while being worthless:

- **Floor 1, the base rate.** Grab 200 eligible pages at random and about 123 of them are already declining. Any precision@200 near 0.62 is a coin flip in a suit.
- **Floor 2, the best single-signal rule.** Section 5 shows the strongest one-liner, sort by staleness, hitting 0.710 at K=200 and 0.860 at K=100. That, not the base rate, is the number my model has to beat before I say anything.

Note in the table below that the rule's precision is not monotone in K: 0.800 at 50, 0.860 at 100, then back down to 0.710 at 200. Section 5 explains why, and it is not noise.

**Secondary metric: impression-weighted recall@K.** Precision counts pages, and pages are not equal. A queue of 200 tiny pages can score well on precision and protect almost no traffic. So alongside it I will report what share of the total impressions-at-risk the top K actually covers. That is the difference between being right and being useful.

What counts as good, written down now, before any training, so I cannot move the line later: **precision@K above the best rule baseline, under a client-holdout split, and still standing under a time-based split in the warehouse.** If it beats the rule on random splits but not on grouped ones, that is a client-memorisation result and I will report it as a failure.

In [4]:
# The metric, computed today on a baseline. If I can't compute it now, it isn't a metric yet.
def precision_at_k(frame, score, k, label="y_demand_loss_30d"):
    top = frame.assign(_s=score).nlargest(k, "_s")
    return top[label].mean()


def impression_recall_at_k(frame, score, k, label="y_demand_loss_30d"):
    at_risk = frame.loc[frame[label] == 1, "impressions_prev_30d"].sum()
    top = frame.assign(_s=score).nlargest(k, "_s")
    caught = top.loc[top[label] == 1, "impressions_prev_30d"].sum()
    return caught / at_risk


staleness = work["days_since_last_update"]

print(f"Base rate (the random-guessing floor): {base_rate * 100:.1f}%")
print()
print(f"{'K':>6}  {'precision@K':>12}  {'lift vs base':>13}  {'impression recall@K':>20}")
for k in (50, 100, 200):
    p = precision_at_k(work, staleness, k)
    r = impression_recall_at_k(work, staleness, k)
    print(f"{k:>6}  {p:>12.3f}  {p / base_rate:>12.2f}x  {r * 100:>19.2f}%")

print()
print("both numbers computable today on a one-line rule, so the metric is real, not aspirational.")
print("note how small impression recall is: a count-based queue protects very little traffic.")

Base rate (the random-guessing floor): 61.5%

     K   precision@K   lift vs base   impression recall@K
    50         0.800          1.30x                 0.35%
   100         0.860          1.40x                 0.68%
   200         0.710          1.15x                 1.07%

both numbers computable today on a one-line rule, so the metric is real, not aspirational.
note how small impression recall is: a count-based queue protects very little traffic.


## 4. The unit of analysis, as a real dataframe

**One row is one page, at one client, summarised over one 90-day window.** The id is `content_id`, and `client_id` is the group it sits in. Not one row per client, not one row per keyword, not one row per day.

That choice follows straight from the decision. The editor opens a page. So the thing I score has to be the thing they act on, and anything coarser (score the client) or finer (score a query) produces a list nobody can work down.

Two consequences I have to carry through the whole project:

- **`client_id` is the grouping key, never a feature.** Pages from one client share a site, a template and an author. If a client appears in both train and test, the model can memorise the client instead of learning the pattern, and my score is fiction. Client-holdout splits from here on.
- **In the capstone the grain gains a date.** On the warehouse daily table the unit becomes one page **at one cutoff date T**, so the same page shows up at several values of T. That is more training rows, but it also means my splits have to be grouped by client *and* respect time, or the same page leaks across the split wearing two different hats.

The cells below show the actual dataframe, check the grain is what I claim it is, and attach the sketched target so the unit and the label are visible in the same table.

In [5]:
# The grain, checked rather than assumed.
dupes = df.duplicated(subset=["content_id"]).sum()
per_client = df.groupby("client_id").size()

print(f"Duplicate content_id rows: {dupes}   (0 means one row really is one page)")
print(f"Pages per client: min {per_client.min():,}, median {per_client.median():,.0f}, max {per_client.max():,}")
print(f"Clients: {df['client_id'].nunique()}  ->  the grouping key for every split I do")
print()

unit_cols = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_prev_30d",
    "impressions_last_30d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "y_demand_loss_30d",
]
print("one row = one page at one client over one 90-day window:")
work[unit_cols].head(8)

Duplicate content_id rows: 0   (0 means one row really is one page)
Pages per client: min 3, median 567, max 7,008
Clients: 32  ->  the grouping key for every split I do

one row = one page at one client over one 90-day window:


,content_id,client_id,content_type,impressions_prev_30d,impressions_last_30d,days_since_last_update,avg_position,ctr,y_demand_loss_30d
0,content_304f48230142,client_f369cb89fc,keyword article,987,578,20,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,5915,2501,25,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,6089,2382,20,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,keyword article,4206,3626,22,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,6452,4211,14,44.0,0.13,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,1009,617,20,8.5,0.03,1
7,content_a63219c6e95a,client_19581e27de,keyword article,632,636,22,21.2,0.06,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,13828,5696,20,46.0,0.09,1


In [6]:
# Same table, but showing the two facts that would bite a blind fillna(0).
print("avg_position == 0 means NO POSITION DATA, not rank zero:")
print(f"  {(df['avg_position'] == 0).sum():,} rows ({(df['avg_position'] == 0).mean() * 100:.1f}%)")
print()
print("keyword data is missing along content_type lines, so fillna(0) would encode the type:")
missing_by_type = df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean())
for content_type, share in missing_by_type.items():
    print(f"  {content_type:20s} {share * 100:>6.1f}% missing search_volume")
print()
print("plan: has_keyword_data / has_position flags instead of filling, decided at the contract step.")

avg_position == 0 means NO POSITION DATA, not rank zero:
  1,205 rows (4.0%)

keyword data is missing along content_type lines, so fillna(0) would encode the type:
  comparison article      0.0% missing search_volume
  feedly article        100.0% missing search_volume
  keyword article         1.4% missing search_volume

plan: has_keyword_data / has_position flags instead of filling, decided at the contract step.


## 5. Why ML beats a fixed rule here

This is the section I most wanted to be wrong about, because if a one-line rule works then the honest answer is to ship the rule and stop. So I tested five rules an experienced editor might genuinely propose, and ranked the same 20,249 pages by each of them.

**Evidence 1: the sensible rules pick almost completely different pages.** Out of 200 slots, "oldest update" and "most impressions" agree on **1 page**. "Most impressions" and "lowest ctr" agree on **zero**. The only pair that overlaps meaningfully is ctr and engagement, and those are close to being the same measurement twice. Five defensible rules, five near-disjoint queues. If the answer were a threshold, they would not disagree this hard.

**Evidence 2: two of the five are worse than doing nothing.** Sorting by "most impressions" scores 0.395 and "worst position" 0.365, against a 0.615 base rate. Both of those sound like reasonable editorial instincts, and both actively hurt. That is the part I would never have guessed from the outside, and it is the strongest argument that this pattern is not available by intuition.

**Evidence 3: a single signal often cannot even produce an order.** This one I found by accident, when the staleness rule scored 0.800 at K=50, 0.860 at K=100, then dropped to 0.710 at K=200. `days_since_last_update` takes only **43 distinct values** across 20,249 pages, and **7,559 pages are tied at the value where the top 200 gets cut**. So "sort by staleness" does not actually rank anything at the depth an editor works at, it produces a bucket 7,559 pages deep and whatever the sort happens to do inside it. The precision wobble is that arbitrariness showing up as a number. A model that combines several signals produces a genuinely continuous score, which is a real advantage before you even compare accuracy.

**Evidence 4: the signals interact, so no single ordering exists.** In the last table the decline rate goes *up* as position gets better: pages in the top 3 decline at about 80%, pages buried deep at about 34%. Backwards from what I expected, and it means "fix the worst-ranked pages first" is exactly the wrong sort order. It also shifts with staleness, so the right threshold for one group is the wrong one for another. That combination, several signals each conditional on the others, is the case where a learned model earns its place instead of just sounding modern.

**The honest counterweight.** Staleness alone gets 0.710 at K=200 and 0.860 at K=100, which is a real bar. So my claim is not "ML wins", it is "ML has to clear the rule at the K that matches real editor capacity, under a client-holdout split, before I am allowed to say anything". If it lands within a couple of points I will report that as a tie and recommend the rule, because a rule an editor can read beats a model they cannot for a two-point gain. That is the outcome I would actually want to defend in Week 7.

A caution on the table: the bottom staleness rows have fewer than 10 pages in some cells, so I print counts beside it and I am reading only the two well-populated rows. A 90% cell built on 10 pages is noise wearing a percentage sign.

In [7]:
# Evidence 1 and 2: five rules an editor might propose, scored on the same pages.
K = 200
rules = {
    "oldest update":     work["days_since_last_update"],
    "most impressions":  work["impressions_90d"],
    "worst position":    work["avg_position"].replace(0, np.nan),
    "lowest ctr":        -work["ctr"],
    "lowest engagement": -work["engagement_rate"],
}

tops = {}
print(f"{'rule':>18}  {'precision@' + str(K):>13}  verdict")
for name, score in rules.items():
    top_idx = score.dropna().nlargest(K).index
    tops[name] = set(top_idx)
    p = work.loc[top_idx, "y_demand_loss_30d"].mean()
    verdict = "beats" if p > base_rate else "WORSE THAN"
    print(f"{name:>18}  {p:>13.3f}  {verdict} the {base_rate:.3f} base rate")

print()
print(f"how many of the same {K} pages does each pair pick?")
names = list(rules)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        print(f"  {a:>18} vs {b:<18} {len(tops[a] & tops[b]):>4} / {K}")

              rule  precision@200  verdict
     oldest update          0.710  beats the 0.615 base rate
  most impressions          0.395  WORSE THAN the 0.615 base rate
    worst position          0.365  WORSE THAN the 0.615 base rate
        lowest ctr          0.640  beats the 0.615 base rate
 lowest engagement          0.655  beats the 0.615 base rate

how many of the same 200 pages does each pair pick?
       oldest update vs most impressions      1 / 200
       oldest update vs worst position        2 / 200
       oldest update vs lowest ctr           26 / 200
       oldest update vs lowest engagement    48 / 200
    most impressions vs worst position        1 / 200
    most impressions vs lowest ctr            0 / 200
    most impressions vs lowest engagement     0 / 200
      worst position vs lowest ctr            8 / 200
      worst position vs lowest engagement     1 / 200
          lowest ctr vs lowest engagement    64 / 200


In [8]:
# Evidence 3: the best rule can't actually produce an order this deep into the list.
best = work["days_since_last_update"]
cutoff_value = best.nlargest(K).min()
tied = int((best == cutoff_value).sum())

print(f"distinct values of days_since_last_update: {best.nunique()} across {len(work):,} pages")
print(f"value where the top {K} gets cut:           {cutoff_value}")
print(f"pages tied at exactly that value:           {tied:,}")
print()
print(f"so 'sort by staleness' hands the editor a {tied:,}-page tie and the top {K} is whichever")
print("ones the sort happened to touch. that's why its precision wobbles with K rather than decaying.")

distinct values of days_since_last_update: 43 across 20,249 pages
value where the top 200 gets cut:           104
pages tied at exactly that value:           7,559

so 'sort by staleness' hands the editor a 7,559-page tie and the top 200 is whichever
ones the sort happened to touch. that's why its precision wobbles with K rather than decaying.


In [9]:
# Evidence 3: the signals interact, so one threshold cannot order the list.
work["staleness_band"] = pd.cut(
    work["days_since_last_update"],
    [-1, 90, 180, 365, 10**9],
    labels=["<=90d", "91-180d", "181-365d", "365d+"],
)

rate = work.pivot_table(
    index="staleness_band", columns="position_tier",
    values="y_demand_loss_30d", aggfunc="mean", observed=True,
) * 100
count = work.pivot_table(
    index="staleness_band", columns="position_tier",
    values="y_demand_loss_30d", aggfunc="size", observed=True,
)

print("decline rate %, by how stale the page is and where it ranks:")
print(rate.round(1).to_string())
print()
print("pages behind each cell (read only the cells with real volume):")
print(count.to_string())
print()
print("top_3 declines ~80% while deep declines ~34%. 'fix the worst-ranked first' is backwards here.")

decline rate %, by how stale the page is and where it ranks:
position_tier   deep  page_1  page_3_5  striking  top_3
staleness_band                                         
<=90d           34.3    60.4      59.7      64.0   79.6
91-180d         41.0    63.0      62.0      64.4   74.1
181-365d         0.0    90.0      85.7      90.0    NaN

pages behind each cell (read only the cells with real volume):
position_tier    deep  page_1  page_3_5  striking  top_3
staleness_band                                          
<=90d           472.0  5273.0    3020.0    3524.0  245.0
91-180d         188.0  2794.0    2518.0    1943.0  243.0
181-365d          2.0    10.0       7.0      10.0    NaN

top_3 declines ~80% while deep declines ~34%. 'fix the worst-ranked first' is backwards here.


## What I am carrying into ML-04

Three things this notebook settled, and one it opened:

1. The task is **ranking**, evaluated as **precision@K** against the **best single-signal rule (0.710 at K=200)**, not against the base rate. And K has to be pinned to real editor capacity, because the rule's score moves around with it.
2. The target is a **forward-window demand loss**, and it **cannot be built honestly on the starter file**, because the 90-day feature columns contain the outcome window. That moves me onto the warehouse daily table sooner than I expected.
3. The unit is **one page at one cutoff date**, grouped by client for every split.

The open one: my eligibility floor of 50 impressions is a number I chose, not one I derived. It decides who is even in the queue, so it deserves a sensitivity check in the data contract rather than a shrug.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.